In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

In [2]:
df = pd.read_csv("PJME_preprocessed.csv")

In [6]:
df[
    [
        "Datetime",
        "PJME_MW",
        "Hour",
        "DayOfWeek",
        "Month",
        "IsWeekend"
    ]
].head(10)

,Datetime,PJME_MW,Hour,DayOfWeek,Month,IsWeekend
0,2002-01-08 01:00:00,29445.0,1,1,1,0
1,2002-01-08 02:00:00,28670.0,2,1,1,0
2,2002-01-08 03:00:00,28375.0,3,1,1,0
3,2002-01-08 04:00:00,28542.0,4,1,1,0
4,2002-01-08 05:00:00,29261.0,5,1,1,0
5,2002-01-08 06:00:00,31348.0,6,1,1,0
6,2002-01-08 07:00:00,35335.0,7,1,1,0
7,2002-01-08 08:00:00,37841.0,8,1,1,0
8,2002-01-08 09:00:00,37417.0,9,1,1,0
9,2002-01-08 10:00:00,36824.0,10,1,1,0


In [9]:
features = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend"
]

In [10]:
df = df.dropna().reset_index(drop=True)

In [11]:
n = len(df)

test_size = int(n * 0.20)

test_start = n - test_size

validation_hours = 60 * 24

validation_start = test_start - validation_hours

In [12]:
train_df = df.iloc[
    :validation_start
].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[
    test_start:
].copy()

In [13]:
feature_columns = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend"
]

In [14]:
X_train_raw = train_df[
    feature_columns
].values

X_validation_raw = validation_df[
    feature_columns
].values

X_test_raw = test_df[
    feature_columns
].values

In [15]:
y_train_raw = train_df[
    ["PJME_MW"]
].values

y_validation_raw = validation_df[
    ["PJME_MW"]
].values

y_test_raw = test_df[
    ["PJME_MW"]
].values

In [16]:
feature_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler.transform(
    X_test_raw
)

In [17]:
target_scaler = MinMaxScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler.transform(
    y_validation_raw
)

y_test_scaled = target_scaler.transform(
    y_test_raw
)

In [18]:
LOOKBACK = 48
HORIZON = 24

In [19]:
def create_sequences(
    X,
    y,
    lookback,
    horizon
):
    
    X_sequences = []
    y_sequences = []

    for i in range(
        lookback,
        len(X) - horizon + 1
    ):
        
        X_sequences.append(
            X[i - lookback:i]
        )
        
        y_sequences.append(
            y[i:i + horizon, 0]
        )

    return (
        np.array(X_sequences),
        np.array(y_sequences)
    )

In [20]:
X_train, y_train = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [21]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

In [22]:
validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [23]:
X_validation, y_validation = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [24]:
rnn_time = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, 7)
    ),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [25]:
rnn_time.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [26]:
history_time = rnn_time.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(
        X_validation,
        y_validation
    ),
    shuffle=False,
    verbose=1
)

Epoch 1/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 27s 13ms/step - loss: 0.0076 - mae: 0.0638 - val_loss: 0.0097 - val_mae: 0.0818
Epoch 2/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0055 - mae: 0.0555 - val_loss: 0.0073 - val_mae: 0.0711
Epoch 3/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0060 - mae: 0.0582 - val_loss: 0.0091 - val_mae: 0.0794
Epoch 4/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0064 - mae: 0.0599 - val_loss: 0.0069 - val_mae: 0.0691
Epoch 5/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0062 - mae: 0.0591 - val_loss: 0.0059 - val_mae: 0.0623
Epoch 6/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0050 - mae: 0.0526 - val_loss: 0.0056 - val_mae: 0.0601
Epoch 7/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0046 - mae: 0.0503 - val_loss: 0.0065 - val_mae: 0.0632
Epoch 8/15
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0036 - mae: 0.0451 - val_loss: 0.0060 - val_mae: 0.0609
Epoch 9/15
1792/1792 ━━━━

In [29]:
y_pred_scaled = rnn_time.predict(
    X_validation,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [30]:
y_validation_actual = target_scaler.inverse_transform(
    y_validation.reshape(-1, 1)
).reshape(
    y_validation.shape
)

In [31]:
y_pred_actual = target_scaler.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).reshape(
    y_pred_scaled.shape
)

In [32]:
mae = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
) * 100
r2 = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [33]:
results_time = pd.DataFrame({
    "Experiment": [
        "RNN-48 + Time Features"
    ],
    "MAE": [mae],
    "RMSE": [rmse],
    "MAPE": [mape],
    "R2": [r2],
    "Bias": [bias]
})
results_time

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632


In [34]:
baseline = pd.DataFrame({
    "Experiment": [
        "Phase 5 RNN-48 Baseline"
    ],
    "MAE": [2213.295950],
    "RMSE": [3018.200386],
    "MAPE": [6.799618],
    "R2": [0.783598],
    "Bias": [-1181.449405]
})

In [35]:
comparison = pd.concat(
    [
        baseline,
        results_time
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632


In [36]:
df["Lag_1"] = df["PJME_MW"].shift(1)

df["Lag_24"] = df["PJME_MW"].shift(24)

df["Lag_48"] = df["PJME_MW"].shift(48)

df["Lag_168"] = df["PJME_MW"].shift(168)

In [37]:
df[
    [
        "Datetime",
        "PJME_MW",
        "Lag_1",
        "Lag_24",
        "Lag_48",
        "Lag_168"
    ]
].head(180)

,Datetime,PJME_MW,Lag_1,Lag_24,Lag_48,Lag_168
0,2002-01-08 01:00:00,29445.0,NaN,NaN,NaN,NaN
1,2002-01-08 02:00:00,28670.0,29445.0,NaN,NaN,NaN
2,2002-01-08 03:00:00,28375.0,28670.0,NaN,NaN,NaN
3,2002-01-08 04:00:00,28542.0,28375.0,NaN,NaN,NaN
4,2002-01-08 05:00:00,29261.0,28542.0,NaN,NaN,NaN
...,...,...,...,...,...,...
175,2002-01-15 08:00:00,34375.0,32057.0,35194.0,25970.0,37841.0
176,2002-01-15 09:00:00,34143.0,34375.0,34939.0,27369.0,37417.0
177,2002-01-15 10:00:00,33509.0,34143.0,34515.0,28455.0,36824.0
178,2002-01-15 11:00:00,33178.0,33509.0,34062.0,28982.0,36504.0


In [38]:
df = df.dropna().reset_index(drop=True)

In [39]:
feature_columns_lag = [
    "PJME_MW",
    
    "Hour_sin",
    "Hour_cos",
    
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    
    "Month",
    "IsWeekend",
    
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168"
]

In [40]:
n = len(df)

test_size = int(n * 0.20)

test_start = n - test_size

validation_hours = 60 * 24

validation_start = test_start - validation_hours

In [41]:
train_df = df.iloc[
    :validation_start
].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[
    test_start:
].copy()

In [42]:
X_train_raw = train_df[
    feature_columns_lag
].values

X_validation_raw = validation_df[
    feature_columns_lag
].values

X_test_raw = test_df[
    feature_columns_lag
].values

In [43]:
y_train_raw = train_df[
    ["PJME_MW"]
].values

y_validation_raw = validation_df[
    ["PJME_MW"]
].values

y_test_raw = test_df[
    ["PJME_MW"]
].values

In [44]:
feature_scaler_lag = MinMaxScaler()

X_train_scaled = feature_scaler_lag.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_lag.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_lag.transform(
    X_test_raw
)

In [45]:
target_scaler_lag = MinMaxScaler()

y_train_scaled = target_scaler_lag.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_lag.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_lag.transform(
    y_test_raw
)

In [46]:
LOOKBACK = 48
HORIZON = 24

In [47]:
X_train_lag, y_train_lag = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [48]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [49]:
X_validation_lag, y_validation_lag = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [50]:
rnn_lag = Sequential([
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, 11)
    ),
    Dense(
        64,
        activation="relu"
    ),  
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [51]:
rnn_lag.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [52]:
history_lag = rnn_lag.fit(
    X_train_lag,
    y_train_lag,
    epochs=15,
    batch_size=64,
    validation_data=(
        X_validation_lag,
        y_validation_lag
    ),
    shuffle=False,
    verbose=1
)

Epoch 1/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 22s 11ms/step - loss: 0.0069 - mae: 0.0605 - val_loss: 0.0103 - val_mae: 0.0849
Epoch 2/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0053 - mae: 0.0546 - val_loss: 0.0110 - val_mae: 0.0885
Epoch 3/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.0060 - mae: 0.0582 - val_loss: 0.0092 - val_mae: 0.0806
Epoch 4/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 0.0067 - mae: 0.0624 - val_loss: 0.0079 - val_mae: 0.0742
Epoch 5/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0062 - mae: 0.0600 - val_loss: 0.0070 - val_mae: 0.0693
Epoch 6/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 0.0057 - mae: 0.0570 - val_loss: 0.0062 - val_mae: 0.0657
Epoch 7/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0050 - mae: 0.0531 - val_loss: 0.0056 - val_mae: 0.0617
Epoch 8/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0046 - mae: 0.0507 - val_loss: 0.0053 - val_mae: 0.0594
Epoch 9/15
1790/1790 ━━━

In [53]:
y_pred_scaled = rnn_lag.predict(
    X_validation_lag,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [54]:
y_validation_actual = target_scaler_lag.inverse_transform(
    y_validation_lag.reshape(-1, 1)
).reshape(
    y_validation_lag.shape
)

y_pred_actual = target_scaler_lag.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).reshape(
    y_pred_scaled.shape
)

In [55]:
mae_lag = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_lag = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_lag = np.sqrt(mse_lag)

mape_lag = mean_absolute_percentage_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
) * 100

r2_lag = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_lag = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [56]:
lag_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + Time + Lag Features"
    ],
    
    "MAE": [mae_lag],
    
    "RMSE": [rmse_lag],
    
    "MAPE": [mape_lag],
    
    "R2": [r2_lag],
    
    "Bias": [bias_lag]
})

lag_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104


In [57]:
comparison = pd.concat(
    [
        comparison,
        lag_result
    ],
    ignore_index=True
)

In [58]:
df["RollingMean_24"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["RollingMean_168"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

df["RollingStd_24"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=24)
    .std()
)

df["RollingStd_168"] = (
    df["PJME_MW"]
    .shift(1)
    .rolling(window=168)
    .std()
)

In [59]:
df[
    [
        "Datetime",
        "PJME_MW",
        "RollingMean_24",
        "RollingMean_168",
        "RollingStd_24",
        "RollingStd_168"
    ]
].tail(20)

,Datetime,PJME_MW,RollingMean_24,RollingMean_168,RollingStd_24,RollingStd_168
145036,2018-08-02 05:00:00,29854.0,39803.833333,35786.809524,6965.040633,6786.448887
145037,2018-08-02 06:00:00,31197.0,39861.041667,35800.077381,6873.026278,6772.561274
145038,2018-08-02 07:00:00,33182.0,39909.458333,35813.125000,6804.549431,6761.494335
145039,2018-08-02 08:00:00,35645.0,39943.500000,35825.994048,6767.105765,6754.370726
145040,2018-08-02 09:00:00,37810.0,39981.166667,35839.607143,6739.347844,6751.670950
145041,2018-08-02 10:00:00,39902.0,40024.708333,35855.708333,6721.016198,6753.133834
145042,2018-08-02 11:00:00,42189.0,40080.125000,35874.970238,6713.996874,6760.070306
145043,2018-08-02 12:00:00,43954.0,40158.166667,35895.976190,6727.729404,6774.244281
145044,2018-08-02 13:00:00,45372.0,40222.791667,35916.255952,6757.613188,6793.324049
145045,2018-08-02 14:00:00,46534.0,40284.666667,35934.392857,6799.292698,6814.570888


In [60]:
df = df.dropna().reset_index(drop=True)

In [61]:
feature_columns_rolling = [
    "PJME_MW",
    
    "Hour_sin",
    "Hour_cos",
    
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    
    "Month",
    "IsWeekend",
    
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "RollingStd_168"
]

In [62]:
n = len(df)

test_size = int(n * 0.20)

test_start = n - test_size

validation_hours = 60 * 24

validation_start = test_start - validation_hours

In [63]:
train_df = df.iloc[
    :validation_start
].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[
    test_start:
].copy()

In [64]:
X_train_raw = train_df[
    feature_columns_rolling
].values

X_validation_raw = validation_df[
    feature_columns_rolling
].values

X_test_raw = test_df[
    feature_columns_rolling
].values

In [65]:
y_train_raw = train_df[
    ["PJME_MW"]
].values

y_validation_raw = validation_df[
    ["PJME_MW"]
].values

y_test_raw = test_df[
    ["PJME_MW"]
].values

In [66]:
feature_scaler_rolling = MinMaxScaler()

X_train_scaled = feature_scaler_rolling.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_rolling.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_rolling.transform(
    X_test_raw
)

In [67]:
target_scaler_rolling = MinMaxScaler()

y_train_scaled = target_scaler_rolling.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_rolling.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_rolling.transform(
    y_test_raw
)

In [68]:
X_train_rolling, y_train_rolling = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [69]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [70]:
X_validation_rolling, y_validation_rolling = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [71]:
rnn_rolling = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, 15)
    ),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [72]:
rnn_rolling.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [73]:
history_rolling = rnn_rolling.fit(
    X_train_rolling,
    y_train_rolling,
    epochs=15,
    batch_size=64,
    validation_data=(
        X_validation_rolling,
        y_validation_rolling
    ),
    shuffle=False,
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 16ms/step - loss: 0.0065 - mae: 0.0591 - val_loss: 0.0095 - val_mae: 0.0813
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0039 - mae: 0.0470 - val_loss: 0.0078 - val_mae: 0.0737
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0039 - mae: 0.0468 - val_loss: 0.0093 - val_mae: 0.0825
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0038 - mae: 0.0467 - val_loss: 0.0088 - val_mae: 0.0801
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - loss: 0.0036 - mae: 0.0456 - val_loss: 0.0062 - val_mae: 0.0670
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 0.0031 - mae: 0.0423 - val_loss: 0.0054 - val_mae: 0.0618
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0027 - mae: 0.0391 - val_loss: 0.0042 - val_mae: 0.0541
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0024 - mae: 0.0361 - val_loss: 0.0037 - val_mae: 0.0493
Epoch 9/15
1788/1788 ━━━

In [74]:
y_pred_scaled = rnn_rolling.predict(
    X_validation_rolling,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [75]:
y_validation_actual = (
    target_scaler_rolling
    .inverse_transform(
        y_validation_rolling.reshape(-1, 1)
    )
    .reshape(y_validation_rolling.shape)
)

y_pred_actual = (
    target_scaler_rolling
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [76]:
mae_rolling = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_rolling = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_rolling = np.sqrt(mse_rolling)

mape_rolling = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_rolling = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_rolling = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [77]:
rolling_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + Time + Lag + Rolling Features"
    ],
    
    "MAE": [mae_rolling],
    
    "RMSE": [rmse_rolling],
    
    "MAPE": [mape_rolling],
    
    "R2": [r2_rolling],
    
    "Bias": [bias_rolling]
})

rolling_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363


In [79]:
comparison = pd.concat(
    [
        comparison,
        rolling_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363


In [80]:
comparison.sort_values(
    by="R2",
    ascending=False
)
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    RNN-48 + Time + Lag + Rolling Features
MAE                                      2045.282073
RMSE                                     2569.550332
MAPE                                        6.362701
R2                                          0.799563
Bias                                      614.004363
Name: 3, dtype: object


In [81]:
best_features = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "RollingStd_168"
]

In [82]:
train_df = df.iloc[:validation_start].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[test_start:].copy()

In [83]:
X_train_raw = train_df[best_features].values

X_validation_raw = validation_df[best_features].values

X_test_raw = test_df[best_features].values

In [84]:
y_train_raw = train_df[["PJME_MW"]].values

y_validation_raw = validation_df[["PJME_MW"]].values

y_test_raw = test_df[["PJME_MW"]].values

In [85]:
feature_scaler_dropout = MinMaxScaler()

X_train_scaled = feature_scaler_dropout.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_dropout.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_dropout.transform(
    X_test_raw
)

In [86]:
target_scaler_dropout = MinMaxScaler()

y_train_scaled = target_scaler_dropout.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_dropout.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_dropout.transform(
    y_test_raw
)

In [87]:
X_train_dropout, y_train_dropout = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [88]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [89]:
X_validation_dropout, y_validation_dropout = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [90]:
n_features = X_train_dropout.shape[2]

print("Number of features:", n_features)

Number of features: 15


In [91]:
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

In [92]:
rnn_dropout = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [93]:
rnn_dropout.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [94]:
history_dropout = rnn_dropout.fit(
    X_train_dropout,
    y_train_dropout,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_dropout,
        y_validation_dropout
    ),
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 16ms/step - loss: 0.0138 - mae: 0.0854 - val_loss: 0.0093 - val_mae: 0.0799
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0070 - mae: 0.0650 - val_loss: 0.0074 - val_mae: 0.0729
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.0055 - mae: 0.0573 - val_loss: 0.0055 - val_mae: 0.0620
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.0046 - mae: 0.0523 - val_loss: 0.0046 - val_mae: 0.0559
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.0043 - mae: 0.0505 - val_loss: 0.0045 - val_mae: 0.0555
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - loss: 0.0042 - mae: 0.0495 - val_loss: 0.0044 - val_mae: 0.0546
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0040 - mae: 0.0483 - val_loss: 0.0041 - val_mae: 0.0522
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0039 - mae: 0.0480 - val_loss: 0.0040 - val_mae: 0.0517
Epoch 9/15
1788/1788 ━━━

In [95]:
y_pred_scaled = rnn_dropout.predict(
    X_validation_dropout,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step


In [96]:
y_validation_actual = (
    target_scaler_dropout
    .inverse_transform(
        y_validation_dropout.reshape(-1, 1)
    )
    .reshape(y_validation_dropout.shape)
)

y_pred_actual = (
    target_scaler_dropout
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [97]:
mae_dropout = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_dropout = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_dropout = np.sqrt(mse_dropout)

mape_dropout = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_dropout = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_dropout = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [98]:
dropout_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + Dropout"
    ],
    "MAE": [mae_dropout],
    "RMSE": [rmse_dropout],
    "MAPE": [mape_dropout],
    "R2": [r2_dropout],
    "Bias": [bias_dropout]
})

In [99]:
comparison = pd.concat(
    [
        comparison,
        dropout_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957


In [100]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104


In [101]:
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    RNN-48 + Time + Lag + Rolling Features
MAE                                      2045.282073
RMSE                                     2569.550332
MAPE                                        6.362701
R2                                          0.799563
Bias                                      614.004363
Name: 3, dtype: object


In [102]:
best_features = [
    "PJME_MW",
    "Hour_sin",
    "Hour_cos",
    "DayOfWeek_sin",
    "DayOfWeek_cos",
    "Month",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "RollingStd_168"
]

In [103]:
train_df = df.iloc[:validation_start].copy()

validation_df = df.iloc[
    validation_start:test_start
].copy()

test_df = df.iloc[test_start:].copy()

In [104]:
X_train_raw = train_df[best_features].values

X_validation_raw = validation_df[best_features].values

X_test_raw = test_df[best_features].values

In [105]:
y_train_raw = train_df[["PJME_MW"]].values

y_validation_raw = validation_df[["PJME_MW"]].values

y_test_raw = test_df[["PJME_MW"]].values

In [106]:
feature_scaler_bn = MinMaxScaler()

X_train_scaled = feature_scaler_bn.fit_transform(
    X_train_raw
)

X_validation_scaled = feature_scaler_bn.transform(
    X_validation_raw
)

X_test_scaled = feature_scaler_bn.transform(
    X_test_raw
)

In [107]:
target_scaler_bn = MinMaxScaler()

y_train_scaled = target_scaler_bn.fit_transform(
    y_train_raw
)

y_validation_scaled = target_scaler_bn.transform(
    y_validation_raw
)

y_test_scaled = target_scaler_bn.transform(
    y_test_raw
)

In [108]:
X_train_bn, y_train_bn = create_sequences(
    X_train_scaled,
    y_train_scaled,
    LOOKBACK,
    HORIZON
)

In [109]:
validation_X_input = np.concatenate([
    X_train_scaled[-LOOKBACK:],
    X_validation_scaled
])

validation_y_input = np.concatenate([
    y_train_scaled[-LOOKBACK:],
    y_validation_scaled
])

In [110]:
X_validation_bn, y_validation_bn = create_sequences(
    validation_X_input,
    validation_y_input,
    LOOKBACK,
    HORIZON
)

In [111]:
n_features = X_train_bn.shape[2]

print("Number of features:", n_features)

Number of features: 15


In [112]:
from tensorflow.keras.layers import (
    SimpleRNN,
    Dense,
    Dropout,
    BatchNormalization
)

In [113]:
rnn_bn = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, n_features)
    ),
    
    BatchNormalization(),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    BatchNormalization(),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [114]:
rnn_bn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [115]:
history_bn = rnn_bn.fit(
    X_train_bn,
    y_train_bn,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_bn,
        y_validation_bn
    ),
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 38s 17ms/step - loss: 0.0866 - mae: 0.1803 - val_loss: 0.0132 - val_mae: 0.0914
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0122 - mae: 0.0877 - val_loss: 0.0147 - val_mae: 0.0946
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0114 - mae: 0.0847 - val_loss: 0.0140 - val_mae: 0.0922
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - loss: 0.0116 - mae: 0.0853 - val_loss: 0.0157 - val_mae: 0.0991
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 36s 20ms/step - loss: 0.0115 - mae: 0.0850 - val_loss: 0.0148 - val_mae: 0.0958
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0118 - mae: 0.0858 - val_loss: 0.0156 - val_mae: 0.0972
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0114 - mae: 0.0847 - val_loss: 0.0142 - val_mae: 0.0957
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0113 - mae: 0.0844 - val_loss: 0.0150 - val_mae: 0.0957
Epoch 9/15
1788/1788 ━━━

In [116]:
y_pred_scaled = rnn_bn.predict(
    X_validation_bn,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


In [117]:
y_validation_actual = (
    target_scaler_bn
    .inverse_transform(
        y_validation_bn.reshape(-1, 1)
    )
    .reshape(y_validation_bn.shape)
)

y_pred_actual = (
    target_scaler_bn
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [118]:
mae_bn = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_bn = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_bn = np.sqrt(mse_bn)

mape_bn = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_bn = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_bn = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [119]:
bn_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + Batch Normalization"
    ],
    "MAE": [mae_bn],
    "RMSE": [rmse_bn],
    "MAPE": [mape_bn],
    "R2": [r2_bn],
    "Bias": [bias_bn]
})

bn_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.04097,-1591.822013


In [120]:
comparison = pd.concat(
    [
        comparison,
        bn_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013


In [121]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013


In [122]:
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    RNN-48 + Time + Lag + Rolling Features
MAE                                      2045.282073
RMSE                                     2569.550332
MAPE                                        6.362701
R2                                          0.799563
Bias                                      614.004363
Name: 3, dtype: object


In [123]:
X_train_optimizer = X_train_bn
y_train_optimizer = y_train_bn

X_validation_optimizer = X_validation_bn
y_validation_optimizer = y_validation_bn

In [124]:
rnn_rmsprop = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [125]:
from tensorflow.keras.optimizers import RMSprop

In [126]:
rmsprop_optimizer = RMSprop(
    learning_rate=0.001
)

In [127]:
rnn_rmsprop.compile(
    optimizer=rmsprop_optimizer,
    loss="mse",
    metrics=["mae"]
)

In [128]:
history_rmsprop = rnn_rmsprop.fit(
    X_train_optimizer,
    y_train_optimizer,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_optimizer,
        y_validation_optimizer
    ),
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 34s 17ms/step - loss: 0.0086 - mae: 0.0653 - val_loss: 0.0077 - val_mae: 0.0728
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 42s 18ms/step - loss: 0.0039 - mae: 0.0472 - val_loss: 0.0072 - val_mae: 0.0690
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 42s 18ms/step - loss: 0.0034 - mae: 0.0443 - val_loss: 0.0068 - val_mae: 0.0673
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.0032 - mae: 0.0431 - val_loss: 0.0063 - val_mae: 0.0653
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0032 - mae: 0.0424 - val_loss: 0.0062 - val_mae: 0.0641
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - loss: 0.0031 - mae: 0.0419 - val_loss: 0.0057 - val_mae: 0.0627
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.0030 - mae: 0.0415 - val_loss: 0.0058 - val_mae: 0.0632
Epoch 8/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0030 - mae: 0.0413 - val_loss: 0.0057 - val_mae: 0.0623
Epoch 9/15
1788/1788 ━━━

In [129]:
y_pred_scaled = rnn_rmsprop.predict(
    X_validation_optimizer,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


In [130]:
y_validation_actual = (
    target_scaler_bn
    .inverse_transform(
        y_validation_optimizer.reshape(-1, 1)
    )
    .reshape(y_validation_optimizer.shape)
)

y_pred_actual = (
    target_scaler_bn
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [131]:
mae_rmsprop = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse_rmsprop = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse_rmsprop = np.sqrt(
    mse_rmsprop
)
mape_rmsprop = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)
r2_rmsprop = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias_rmsprop = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [132]:
rmsprop_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + RMSprop"
    ],
    "MAE": [
        mae_rmsprop
    ],
    "RMSE": [
        rmse_rmsprop
    ],
    "MAPE": [
        mape_rmsprop
    ],
    "R2": [
        r2_rmsprop
    ],
    "Bias": [
        bias_rmsprop
    ]
})

rmsprop_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + RMSprop,2897.98413,3482.218456,9.107735,0.631893,610.634055


In [133]:
comparison = pd.concat(
    [
        comparison,
        rmsprop_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055


In [134]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013


In [135]:
best_previous = comparison.loc[
    comparison["R2"].idxmax()
]

print(best_previous)

Experiment    RNN-48 + Time + Lag + Rolling Features
MAE                                      2045.282073
RMSE                                     2569.550332
MAPE                                        6.362701
R2                                          0.799563
Bias                                      614.004363
Name: 3, dtype: object


In [136]:
X_train_lr = X_train_optimizer
y_train_lr = y_train_optimizer

X_validation_lr = X_validation_optimizer
y_validation_lr = y_validation_optimizer

n_features = X_train_lr.shape[2]

In [137]:
rnn_lr = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [138]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [139]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [147]:
rmsprop_optimizer = RMSprop(
    learning_rate=0.001
)

In [148]:
rnn_lr.compile(
    optimizer=rmsprop_optimizer,
    loss="mse",
    metrics=["mae"]
)

In [149]:
history_lr = rnn_lr.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 31s 16ms/step - loss: 0.0081 - mae: 0.0647 - val_loss: 0.0076 - val_mae: 0.0716 - learning_rate: 0.0010
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0040 - mae: 0.0479 - val_loss: 0.0067 - val_mae: 0.0682 - learning_rate: 0.0010
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0036 - mae: 0.0458 - val_loss: 0.0065 - val_mae: 0.0654 - learning_rate: 0.0010
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - loss: 0.0035 - mae: 0.0447 - val_loss: 0.0060 - val_mae: 0.0644 - learning_rate: 0.0010
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - loss: 0.0033 - mae: 0.0436 - val_loss: 0.0061 - val_mae: 0.0640 - learning_rate: 0.0010
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - loss: 0.0032 - mae: 0.0428 - val_loss: 0.0060 - val_mae: 0.0638 - learning_rate: 0.0010
Epoch 7/15
1785/1788 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0032 - mae: 0.0425
Epoch 7: ReduceLROnPlateau reducing lear

In [150]:
y_pred_scaled = rnn_lr.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [151]:
y_validation_actual = (
    target_scaler_bn
    .inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [152]:
mae_lr = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse_lr = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse_lr = np.sqrt(mse_lr)
mape_lr = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)
r2_lr = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias_lr = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [153]:
lr_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + RMSprop + LR Scheduling"
    ],
    "MAE": [mae_lr],
    "RMSE": [rmse_lr],
    "MAPE": [mape_lr],
    "R2": [r2_lr],
    "Bias": [bias_lr]
})

lr_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.50112,0.783514,123.542595


In [154]:
comparison = pd.concat(
    [
        comparison,
        lr_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055
8,RNN-48 + RMSprop + LR Scheduling,18707.087116,22432.246512,56.229193,-14.275946,-17283.056916
9,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.501120,0.783514,123.542595


In [155]:
X_train_lr = X_train_optimizer
y_train_lr = y_train_optimizer

X_validation_lr = X_validation_optimizer
y_validation_lr = y_validation_optimizer

n_features = X_train_lr.shape[2]

In [156]:
from tensorflow.keras.callbacks import EarlyStopping

In [157]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [158]:
rnn_early = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [159]:
optimizer = RMSprop(
    learning_rate=0.001
)

In [160]:
rnn_early.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)

In [161]:
history_early = rnn_early.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler,
        early_stopping
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 29s 15ms/step - loss: 0.0082 - mae: 0.0648 - val_loss: 0.0083 - val_mae: 0.0755 - learning_rate: 0.0010
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0039 - mae: 0.0476 - val_loss: 0.0077 - val_mae: 0.0703 - learning_rate: 0.0010
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0036 - mae: 0.0454 - val_loss: 0.0067 - val_mae: 0.0664 - learning_rate: 0.0010
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.0034 - mae: 0.0439 - val_loss: 0.0064 - val_mae: 0.0652 - learning_rate: 0.0010
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - loss: 0.0032 - mae: 0.0428 - val_loss: 0.0061 - val_mae: 0.0645 - learning_rate: 0.0010
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - loss: 0.0031 - mae: 0.0421 - val_loss: 0.0058 - val_mae: 0.0631 - learning_rate: 0.0010
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - loss: 0.0031 - mae: 0.0417 - val_loss: 0.0057 - val_mae: 0.0618 - 

In [162]:
y_pred_scaled = rnn_early.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [163]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [164]:
mae_early = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_early = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_early = np.sqrt(mse_early)

mape_early = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_early = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_early = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [165]:
early_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + RMSprop + LR Scheduling + Early Stopping"
    ],
    "MAE": [mae_early],
    "RMSE": [rmse_early],
    "MAPE": [mape_early],
    "R2": [r2_early],
    "Bias": [bias_early]
})

early_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + RMSprop + LR Scheduling + Early Stopping,2385.324975,2990.615787,7.351098,0.728491,154.019297


In [166]:
comparison = pd.concat(
    [
        comparison,
        early_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055
8,RNN-48 + RMSprop + LR Scheduling,18707.087116,22432.246512,56.229193,-14.275946,-17283.056916
9,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.501120,0.783514,123.542595


In [167]:
rnn_batch32 = Sequential([
    
    SimpleRNN(
        64,
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dropout(0.2),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [168]:
optimizer_batch32 = RMSprop(
    learning_rate=0.001
)

In [169]:
rnn_batch32.compile(
    optimizer=optimizer_batch32,
    loss="mse",
    metrics=["mae"]
)

In [170]:
lr_scheduler_batch32 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [171]:
early_stopping_batch32 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [172]:
history_batch32 = rnn_batch32.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=32,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler_batch32,
        early_stopping_batch32
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 53s 14ms/step - loss: 0.0062 - mae: 0.0569 - val_loss: 0.0093 - val_mae: 0.0822 - learning_rate: 0.0010
Epoch 2/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 43s 12ms/step - loss: 0.0035 - mae: 0.0447 - val_loss: 0.0095 - val_mae: 0.0837 - learning_rate: 0.0010
Epoch 3/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 42s 12ms/step - loss: 0.0033 - mae: 0.0429 - val_loss: 0.0091 - val_mae: 0.0821 - learning_rate: 0.0010
Epoch 4/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 42s 12ms/step - loss: 0.0031 - mae: 0.0421 - val_loss: 0.0089 - val_mae: 0.0806 - learning_rate: 0.0010
Epoch 5/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 86s 13ms/step - loss: 0.0030 - mae: 0.0415 - val_loss: 0.0091 - val_mae: 0.0818 - learning_rate: 0.0010
Epoch 6/15
3575/3575 ━━━━━━━━━━━━━━━━━━━━ 44s 12ms/step - loss: 0.0030 - mae: 0.0410 - val_loss: 0.0091 - val_mae: 0.0819 - learning_rate: 0.0010
Epoch 7/15
3572/3575 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0030 - mae: 0.0408
Epoch 7: ReduceLROnPlateau reducing lear

In [173]:
y_pred_scaled = rnn_batch32.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step


In [174]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [175]:
mae_batch32 = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_batch32 = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_batch32 = np.sqrt(mse_batch32)

mape_batch32 = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_batch32 = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_batch32 = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [176]:
batch32_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + RMSprop + LR Scheduling + Early Stopping + Batch Size 32"
    ],
    "MAE": [mae_batch32],
    "RMSE": [rmse_batch32],
    "MAPE": [mape_batch32],
    "R2": [r2_batch32],
    "Bias": [bias_batch32]
})

batch32_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + RMSprop + LR Scheduling + Early Stopp...,2326.750397,2921.702505,7.241754,0.74086,308.128101


In [177]:
comparison = pd.concat(
    [
        comparison,
        batch32_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055
8,RNN-48 + RMSprop + LR Scheduling,18707.087116,22432.246512,56.229193,-14.275946,-17283.056916
9,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.501120,0.783514,123.542595


In [178]:
X_train_lr = X_train_optimizer
y_train_lr = y_train_optimizer

X_validation_lr = X_validation_optimizer
y_validation_lr = y_validation_optimizer

n_features = X_train_lr.shape[2]

In [179]:
rnn_layers = Sequential([
    
    SimpleRNN(
        64,
        return_sequences=True,
        input_shape=(LOOKBACK, n_features)
    ),
    
    Dropout(0.2),
    
    SimpleRNN(
        32
    ),
    
    Dropout(0.2),
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(HORIZON)
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [180]:
optimizer_layers = RMSprop(
    learning_rate=0.001
)

In [181]:
rnn_layers.compile(
    optimizer=optimizer_layers,
    loss="mse",
    metrics=["mae"]
)

In [182]:
lr_scheduler_layers = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [183]:
early_stopping_layers = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [184]:
history_layers = rnn_layers.fit(
    X_train_lr,
    y_train_lr,
    
    epochs=15,
    batch_size=64,
    
    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),
    
    callbacks=[
        lr_scheduler_layers,
        early_stopping_layers
    ],
    
    shuffle=False,
    
    verbose=1
)

Epoch 1/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 67s 33ms/step - loss: 0.0091 - mae: 0.0669 - val_loss: 0.0104 - val_mae: 0.0864 - learning_rate: 0.0010
Epoch 2/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 56s 32ms/step - loss: 0.0034 - mae: 0.0439 - val_loss: 0.0095 - val_mae: 0.0831 - learning_rate: 0.0010
Epoch 3/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 54s 30ms/step - loss: 0.0030 - mae: 0.0410 - val_loss: 0.0086 - val_mae: 0.0778 - learning_rate: 0.0010
Epoch 4/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 54s 30ms/step - loss: 0.0028 - mae: 0.0397 - val_loss: 0.0082 - val_mae: 0.0765 - learning_rate: 0.0010
Epoch 5/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 82s 30ms/step - loss: 0.0027 - mae: 0.0389 - val_loss: 0.0081 - val_mae: 0.0760 - learning_rate: 0.0010
Epoch 6/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 54s 30ms/step - loss: 0.0026 - mae: 0.0381 - val_loss: 0.0076 - val_mae: 0.0730 - learning_rate: 0.0010
Epoch 7/15
1788/1788 ━━━━━━━━━━━━━━━━━━━━ 88s 34ms/step - loss: 0.0025 - mae: 0.0376 - val_loss: 0.0073 - val_mae: 0.0709 - 

In [185]:
y_pred_scaled = rnn_layers.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step


In [186]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [187]:
mae_layers = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

mse_layers = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

rmse_layers = np.sqrt(mse_layers)

mape_layers = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)

r2_layers = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)

bias_layers = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [188]:
layers_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 + 2 Layers (64-32)"
    ],
    "MAE": [mae_layers],
    "RMSE": [rmse_layers],
    "MAPE": [mape_layers],
    "R2": [r2_layers],
    "Bias": [bias_layers]
})

layers_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 + 2 Layers (64-32),3003.001419,3636.364202,9.269019,0.598582,208.944454


In [189]:
comparison = pd.concat(
    [
        comparison,
        layers_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055
8,RNN-48 + RMSprop + LR Scheduling,18707.087116,22432.246512,56.229193,-14.275946,-17283.056916
9,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.501120,0.783514,123.542595


In [190]:
import keras_tuner as kt

In [191]:
def build_rnn_model(hp):

    model = Sequential()

    rnn_units = hp.Choice(
        "rnn_units",
        values=[32, 64, 128]
    )

    dense_units = hp.Choice(
        "dense_units",
        values=[32, 64, 128]
    )

    dropout_rate = hp.Choice(
        "dropout_rate",
        values=[0.1, 0.2, 0.3]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0005, 0.001, 0.002]
    )

    model.add(
        SimpleRNN(
            units=rnn_units,
            input_shape=(LOOKBACK, n_features)
        )
    )

    model.add(
        Dropout(dropout_rate)
    )

    model.add(
        Dense(
            units=dense_units,
            activation="relu"
        )
    )

    model.add(
        Dropout(dropout_rate)
    )

    model.add(
        Dense(HORIZON)
    )

    optimizer = RMSprop(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

In [192]:
tuner = kt.RandomSearch(
    build_rnn_model,
    objective="val_loss",
    max_trials=5,
    executions_per_trial=1,
    directory="rnn_tuning",
    project_name="electricity_demand_rnn",
    overwrite=True
)

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [193]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
rnn_units (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
dense_units (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
dropout_rate (Choice)
{'default': 0.1, 'conditions': [], 'values': [0.1, 0.2, 0.3], 'ordered': True}
learning_rate (Choice)
{'default': 0.0005, 'conditions': [], 'values': [0.0005, 0.001, 0.002], 'ordered': True}


In [194]:
lr_scheduler_tuning = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [195]:
early_stopping_tuning = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [196]:
tuner.search(
    X_train_lr,
    y_train_lr,

    epochs=15,

    batch_size=64,

    validation_data=(
        X_validation_lr,
        y_validation_lr
    ),

    callbacks=[
        lr_scheduler_tuning,
        early_stopping_tuning
    ],

    shuffle=False,

    verbose=1
)

Trial 5 Complete [00h 06m 54s]
val_loss: 0.004627761896699667

Best val_loss So Far: 0.0035772346891462803
Total elapsed time: 00h 42m 13s


In [197]:
best_hp = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

In [198]:
print(
    "Best RNN units:",
    best_hp.get("rnn_units")
)

print(
    "Best Dense units:",
    best_hp.get("dense_units")
)

print(
    "Best Dropout:",
    best_hp.get("dropout_rate")
)

print(
    "Best Learning Rate:",
    best_hp.get("learning_rate")
)

Best RNN units: 128
Best Dense units: 32
Best Dropout: 0.1
Best Learning Rate: 0.0005


In [199]:
best_rnn_tuned = tuner.get_best_models(
    num_models=1
)[0]

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 9 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [200]:
y_pred_scaled = best_rnn_tuned.predict(
    X_validation_lr,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


In [201]:
y_validation_actual = (
    target_scaler_bn.inverse_transform(
        y_validation_lr.reshape(-1, 1)
    )
    .reshape(y_validation_lr.shape)
)

y_pred_actual = (
    target_scaler_bn.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .reshape(y_pred_scaled.shape)
)

In [202]:
mae_tuned = mean_absolute_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
mse_tuned = mean_squared_error(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
rmse_tuned = np.sqrt(mse_tuned)
mape_tuned = (
    mean_absolute_percentage_error(
        y_validation_actual.flatten(),
        y_pred_actual.flatten()
    ) * 100
)
r2_tuned = r2_score(
    y_validation_actual.flatten(),
    y_pred_actual.flatten()
)
bias_tuned = np.mean(
    y_pred_actual.flatten()
    -
    y_validation_actual.flatten()
)

In [203]:
tuned_result = pd.DataFrame({
    "Experiment": [
        "RNN-48 Hyperparameter Tuning"
    ],
    "MAE": [mae_tuned],
    "RMSE": [rmse_tuned],
    "MAPE": [mape_tuned],
    "R2": [r2_tuned],
    "Bias": [bias_tuned]
})

tuned_result

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,RNN-48 Hyperparameter Tuning,2288.766682,2838.880729,7.094536,0.755343,265.951903


In [204]:
comparison = pd.concat(
    [
        comparison,
        tuned_result
    ],
    ignore_index=True
)
comparison

,Experiment,MAE,RMSE,MAPE,R2,Bias
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
2,RNN-48 + Time + Lag Features,2894.430283,3684.343609,8.944598,0.571222,892.884104
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
6,RNN-48 + Batch Normalization,4421.516925,5620.627127,12.949586,0.040970,-1591.822013
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055
8,RNN-48 + RMSprop + LR Scheduling,18707.087116,22432.246512,56.229193,-14.275946,-17283.056916
9,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.501120,0.783514,123.542595


In [205]:
best_index = comparison["R2"].idxmax()

best_rnn_result = comparison.loc[
    best_index
]

print("BEST RNN RESULT")
print(best_rnn_result)

BEST RNN RESULT
Experiment    RNN-48 + Time + Lag + Rolling Features
MAE                                      2045.282073
RMSE                                     2569.550332
MAPE                                        6.362701
R2                                          0.799563
Bias                                      614.004363
Name: 3, dtype: object


In [206]:
comparison.sort_values(
    by="R2",
    ascending=False
)

,Experiment,MAE,RMSE,MAPE,R2,Bias
4,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
3,RNN-48 + Time + Lag + Rolling Features,2045.282073,2569.550332,6.362701,0.799563,614.004363
0,Phase 5 RNN-48 Baseline,2213.295950,3018.200386,6.799618,0.783598,-1181.449405
9,RNN-48 + RMSprop + LR Scheduling,2102.411521,2670.445912,6.501120,0.783514,123.542595
5,RNN-48 + Dropout,2236.203850,2774.639727,7.014936,0.766291,593.137957
13,RNN-48 Hyperparameter Tuning,2288.766682,2838.880729,7.094536,0.755343,265.951903
11,RNN-48 + RMSprop + LR Scheduling + Early Stopp...,2326.750397,2921.702505,7.241754,0.740860,308.128101
10,RNN-48 + RMSprop + LR Scheduling + Early Stopping,2385.324975,2990.615787,7.351098,0.728491,154.019297
1,RNN-48 + Time Features,2338.989995,2961.051034,7.132572,0.714994,-328.877632
7,RNN-48 + RMSprop,2897.984130,3482.218456,9.107735,0.631893,610.634055


In [207]:
comparison.to_csv(
    "phase6_rnn_final_comparison.csv",
    index=False
)

In [208]:
best_rnn_tuned.save(
    "best_rnn_phase6.keras"
)